# Notebook area

This notebook calculates burnt area ratios for each province and territory for each year. It serves as preparation of the data for the following notbook choropleth_map where the interactive choroplet map will be generated.

### Input data
- fire_tot.csv: Cleaned wildfire data for the years 2014 - 2023 including a column with size
- lpr_000b21a_e.zip: Raw province and territory data of Canada

### Outputs
- burnt_area.csv: Wildfire data including for each year and each province the area of the province (in hectare), the area burnt down by wildfire (in hectare) and the percentage of the area burnt down by wildfires
- canada_provinces.gpkg: Provinces and Territories of Canada data with crs EPSG:3347, but column "PRENAME" now as "province"

### Key assumptions
- Province/territory data of Canada in EPSG:3347 for the area calcluation
- Province/territory data of Canada has to rename the column "PRENAME" to "province" to later make the merging easyer

In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
# 1. Loading data
# Load wildfire data
fire_tot = pd.read_csv("../data/processed/fire_tot.csv", sep = ",")
print("Data loaded successfully")

# Load canada data
canada = gpd.read_file("../data/raw/lpr_000b21a_e.zip")
print("Zip loaded successfully")

Data loaded successfully
Zip loaded successfully


In [3]:
# 3. Rename "PRENAME" to "province"
# Rename
print(canada.columns)
canada = canada.rename(columns={"PRENAME": "province"})

# 3. Exporting Canada data with column name "province"
canada.to_file("../data/processed/choropleth_map/canada_provinces.gpkg", driver="GPKG")
print("Export sucessful")

Index(['PRUID', 'DGUID', 'PRNAME', 'PRENAME', 'PRFNAME', 'PREABBR', 'PRFABBR',
       'LANDAREA', 'geometry'],
      dtype='str')
Export sucessful


In [4]:
# 4. Calculate the area of Canada in hectares
canada["area_ha"] = canada.geometry.area / 10000
print(canada[["province", "area_ha"]])

                     province       area_ha
0   Newfoundland and Labrador  3.973420e+07
1        Prince Edward Island  5.862158e+05
2                 Nova Scotia  5.728464e+06
3               New Brunswick  7.419342e+06
4                      Quebec  1.475449e+08
5                     Ontario  9.793295e+07
6                    Manitoba  6.277102e+07
7                Saskatchewan  6.320452e+07
8                     Alberta  6.399287e+07
9            British Columbia  9.172998e+07
10                      Yukon  4.556615e+07
11      Northwest Territories  1.276024e+08
12                    Nunavut  2.009083e+08


In [5]:
# 5. Contolling if the files have the same provinces --> If Nunavut and Prince Edward Island are teh differenc this is not a problem, as they have no wildfire data
set(canada["province"]) - set(fire_tot["province"])

{'Nunavut', 'Prince Edward Island'}

In [6]:
# 6. Calculate burnt area for each province for each year
fire_sum = ( fire_tot.groupby(["year", "province"])["size_ha"].sum().reset_index())

fire_sum.head(12)

,year,province,size_ha
0,2014,Alberta,103663.26
1,2014,British Columbia,363513.61
2,2014,Manitoba,36107.70
3,2014,Newfoundland and Labrador,8832.00
4,2014,Northwest Territories,3597379.41
5,2014,Nova Scotia,200.90
6,2014,Ontario,4917.00
7,2014,Quebec,62271.70
8,2014,Saskatchewan,338952.76
9,2014,Yukon,2901.74


In [7]:
# 7. Merge canada file with calculated area, and fire file with calculated burnt area together
fire_sum = fire_sum.merge(
    canada[["province", "area_ha"]],
    on="province",
    how="left" # keep all rows from fire_sum and merge the data that fits from canada to fire_sum
)

In [8]:
# 8. Calculation of burnt area ratio for each province
fire_sum["burnt_percent"] = (fire_sum["size_ha"] / fire_sum["area_ha"]) * 100

fire_sum.head(5)

,year,province,size_ha,area_ha,burnt_percent
0,2014,Alberta,103663.26,6.399287e+07,0.161992
1,2014,British Columbia,363513.61,9.172998e+07,0.396287
2,2014,Manitoba,36107.70,6.277102e+07,0.057523
3,2014,Newfoundland and Labrador,8832.00,3.973420e+07,0.022228
4,2014,Northwest Territories,3597379.41,1.276024e+08,2.819210


In [9]:
# 9. Exporting
fire_sum.to_csv("../data/processed/choropleth_map/burnt_area.csv", index = False)
print("Export successful")

Export successful
